In [1]:
import time
import re
import pandas as pd
from IPython.display import clear_output

import praw
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

# Plotly for interactive charts
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
#Configuring Reddit API

R_CLIENT_ID = "YOUR_REDDIT_CLIENT_ID"
R_CLIENT_SECRET = "YOUR_REDDIT_CLIENT_SECRET"
R_USER_AGENT = "SocialJusticeSentiment/1.0"

# Multiple subreddits 

STREAM_SUBREDDITS = ["HumanRights", "Palestine", "worldnews", "Israel"]
STOP_WORDS = set(stopwords.words("english"))
VADER = SentimentIntensityAnalyzer()

In [3]:
# Cleaning text
def clean_text(text: str) -> str:
    if not isinstance(text, str) or text.strip() == "":
        return ""

    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in STOP_WORDS]
    return " ".join(tokens)


# Sentiment functions
def vader_sentiment(text: str) -> str:
    score = VADER.polarity_scores(text)["compound"]
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

def tb_polarity(text: str) -> float:
    return TextBlob(text).sentiment.polarity

def tb_subjectivity(text: str) -> float:
    return TextBlob(text).sentiment.subjectivity

def tb_sentiment(text: str) -> str:
    polarity = TextBlob(text).sentiment.polarity
    if polarity >= 0.05:
        return "positive"
    elif polarity <= -0.05:
        return "negative"
    return "neutral"

In [4]:
def create_reddit_client():
    return praw.Reddit(
        client_id=R_CLIENT_ID,
        client_secret=R_CLIENT_SECRET,
        user_agent=R_USER_AGENT,
    )

reddit = create_reddit_client()

# create multi-subreddit string for PRAW library
stream_sub = "+".join(STREAM_SUBREDDITS)

In [5]:
# Setting up live streaming
live_data = []
last_plot_time = time.time()

print(f"Starting real-time monitoring of r/{stream_sub}...")
print("Interactive bar + polarity chart refresh every 100 seconds.")
print("Press STOP / Interrupt in Jupyter to end the stream.\n")

try:
    for comment in reddit.subreddit(stream_sub).stream.comments(
        skip_existing=True,
        pause_after=5
    ):
        if comment is None:
            continue
        
        text = comment.body
        if not text:
            continue

        cleaned = clean_text(text)
        if cleaned == "":
            continue

        vader = vader_sentiment(cleaned)
        polarity = tb_polarity(cleaned)
        subjectivity = tb_subjectivity(cleaned)
        tb_label = tb_sentiment(cleaned)

        live_data.append({
            "comment_id": comment.id,
            "subreddit": str(comment.subreddit),
            "text": text,
            "clean_text": cleaned,
            "vader_sentiment": vader,
            "tb_polarity": polarity,
            "tb_subjectivity": subjectivity,
            "tb_sentiment": tb_label,
            "created_utc": pd.to_datetime(comment.created_utc, unit="s")
        })

        now = time.time()
        if now - last_plot_time >= 100:
            clear_output(wait=True)

            print(f"Real-time monitoring: r/{stream_sub}")
            print(f"Total collected comments: {len(live_data)}")

            df_live = pd.DataFrame(live_data)

            if not df_live.empty:
                # --- VADER sentiment counts (for bar chart) ---
                sentiment_totals = df_live["vader_sentiment"].value_counts().reindex(
                    ["positive", "neutral", "negative"],
                    fill_value=0
                )

                total = sentiment_totals.sum() if sentiment_totals.sum() > 0 else 1
                sentiment_percent = (sentiment_totals / total * 100).round(1)

                color_map = {
                    "positive": "green",
                    "neutral": "gray",
                    "negative": "red"
                }

                # --- Prepare polarity data for rolling average line chart ---
                df_polar = df_live.copy()
                df_polar = df_polar.sort_values("created_utc")
                df_polar["rolling_polarity"] = (
                    df_polar["tb_polarity"]
                    .rolling(window=30, min_periods=5)
                    .mean()
                )

                # Making bar + polarity line charts interactive using subplots
                fig = make_subplots(
                    rows=1,
                    cols=2,
                    specs=[[{"type": "bar"}, {"type": "xy"}]],
                    subplot_titles=(
                        "Live Sentiment (Counts)",
                        "Live TextBlob Polarity (Rolling Average)"
                    )
                )

                # 1️⃣ Plot bar chart (VADER counts)
                fig.add_trace(
                    go.Bar(
                        x=sentiment_totals.index,
                        y=sentiment_totals.values,
                        marker_color=[color_map[s] for s in sentiment_totals.index],
                        text=[
                            f"{c} comments – {p}%"
                            for c, p in zip(
                                sentiment_totals.values,
                                sentiment_percent.values
                            )
                        ],
                        hovertemplate=(
                            "Sentiment: %{x}<br>"
                            "Count: %{y}<br>"
                            "%{text}<extra></extra>"
                        ),
                        name="Sentiment counts"
                    ),
                    row=1, col=1
                )

                # 2️⃣ Plot polarity rolling average line (TextBlob)
                fig.add_trace(
                    go.Scatter(
                        x=df_polar["created_utc"],
                        y=df_polar["rolling_polarity"],
                        mode="lines",
                        name="Rolling polarity",
                        hovertemplate=(
                            "Time: %{x}<br>"
                            "Polarity: %{y:.3f}<extra></extra>"
                        )
                    ),
                    row=1, col=2
                )

                fig.update_xaxes(title_text="Sentiment", row=1, col=1)
                fig.update_yaxes(title_text="Count", row=1, col=1)

                fig.update_xaxes(title_text="Time", row=1, col=2)
                fig.update_yaxes(title_text="Polarity (smoothed)", row=1, col=2)

                fig.update_layout(
                    title=f"Live Sentiment Dashboard – r/{stream_sub}",
                    height=500,
                    width=950,
                    showlegend=False
                )

                fig.show()
                # ---------------------------------------------

            last_plot_time = now

except KeyboardInterrupt:
    print("\nStreaming stopped by user.")
    final_df = pd.DataFrame(live_data)
    print(f"Collected {len(final_df)} comments total.")
    final_df.to_csv("live_reddit_sentiment.csv", index=False)
    print("Saved dataset to live_reddit_sentiment.csv")


Real-time monitoring: r/HumanRights+Palestine+worldnews+Israel
Total collected comments: 194


TooManyRequests: received 429 HTTP response